##  Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix)
from wordcloud import WordCloud

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("✅ All libraries imported successfully!")

In [ ]:
# Cell 2 - Load Dataset
df = pd.read_csv('Tweets.csv')

# Keep useful columns only
df = df[['airline_sentiment', 'text']]
df.columns = ['sentiment', 'text']

# Remove neutral for cleaner 3-class analysis
print("✅ Dataset loaded!")
print(f"Shape: {df.shape}")
print(f"\nSentiment Distribution:")
print(df['sentiment'].value_counts())
df.head()

In [ ]:
# Cell 3 - EDA
plt.figure(figsize=(8, 5))
colors = ['#e74c3c', '#95a5a6', '#2ecc71']
df['sentiment'].value_counts().plot(kind='bar', color=colors, edgecolor='black')
plt.title('Sentiment Distribution', fontsize=15, fontweight='bold')
plt.xlabel('Sentiment', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150)
plt.show()

# Tweet length analysis
df['tweet_length'] = df['text'].apply(len)
plt.figure(figsize=(10, 5))
for sentiment, color in zip(['positive', 'neutral', 'negative'], 
                             ['#2ecc71', '#95a5a6', '#e74c3c']):
    df[df['sentiment']==sentiment]['tweet_length'].plot(
        kind='hist', bins=50, alpha=0.6, color=color, label=sentiment)
plt.title('Tweet Length by Sentiment', fontsize=15, fontweight='bold')
plt.xlabel('Tweet Length')
plt.ylabel('Frequency')
plt.legend()
plt.tight_layout()
plt.savefig('tweet_length.png', dpi=150)
plt.show()

print(f"Average tweet lengths:")
for s in ['positive', 'neutral', 'negative']:
    avg = df[df['sentiment']==s]['tweet_length'].mean()
    print(f"  {s.capitalize()}: {avg:.1f} chars")

In [ ]:
# Cell 4 - WordCloud

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
sentiments = ['positive', 'neutral', 'negative']
colors_map = ['Greens', 'Blues', 'Reds']
titles = ['😊 Positive', '😐 Neutral', '😡 Negative']

for ax, sentiment, cmap, title in zip(axes, sentiments, colors_map, titles):
    text = ' '.join(df[df['sentiment']==sentiment]['text'].values)
    wordcloud = WordCloud(width=600, height=400,
                          background_color='white',
                          colormap=cmap,
                          max_words=100).generate(text)
    ax.imshow(wordcloud, interpolation='bilinear')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('wordcloud.png', dpi=150)
plt.show()
print("✅ WordClouds generated!")

In [ ]:
# Cell 5 - Text Preprocessing
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove mentions & hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    # Remove special characters & numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenize
    words = text.split()
    # Remove stopwords & lemmatize
    words = [lemmatizer.lemmatize(word) for word in words 
             if word not in stop_words and len(word) > 2]
    return ' '.join(words)

df['cleaned_text'] = df['text'].apply(preprocess_text)

print("✅ Text preprocessing done!")
print("\nOriginal tweet:")
print(df['text'][0])
print("\nCleaned tweet:")
print(df['cleaned_text'][0])

In [ ]:
# Cell 6 - Encoding & Split
# Encode labels
label_map = {'positive': 2, 'neutral': 1, 'negative': 0}
df['label'] = df['sentiment'].map(label_map)

X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("✅ Data split done!")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

In [ ]:
# Cell 7 - TF-IDF
tfidf = TfidfVectorizer(max_features=7000, 
                         ngram_range=(1, 2),
                         min_df=2)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("✅ TF-IDF applied!")
print(f"Feature matrix shape: {X_train_tfidf.shape}")

In [ ]:
# Cell 8 - Train Models

# Model 1: Naive Bayes
nb_model = MultinomialNB(alpha=0.1)
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

# Model 2: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, C=5)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

# Model 3: Linear SVM
svm_model = LinearSVC(random_state=42, C=1.0)
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)

print("✅ All 3 models trained!")

In [ ]:
# Cell 9 - Evaluation

def evaluate_model(name, y_true, y_pred):
    print(f"\n{'='*50}")
    print(f"  📌 {name}")
    print(f"{'='*50}")
    print(f"  Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(classification_report(y_true, y_pred, 
          target_names=['Negative', 'Neutral', 'Positive']))

evaluate_model("Naive Bayes",         y_test, nb_pred)
evaluate_model("Logistic Regression", y_test, lr_pred)
evaluate_model("Linear SVM",          y_test, svm_pred)

# Accuracy comparison bar chart
models = ['Naive Bayes', 'Logistic\nRegression', 'Linear SVM']
accuracies = [
    accuracy_score(y_test, nb_pred)*100,
    accuracy_score(y_test, lr_pred)*100,
    accuracy_score(y_test, svm_pred)*100
]

plt.figure(figsize=(8, 5))
bars = plt.bar(models, accuracies, color=['#3498db','#2ecc71','#e67e22'], 
               edgecolor='black', width=0.5)
plt.ylim(60, 100)
plt.title('Model Accuracy Comparison', fontsize=15, fontweight='bold')
plt.ylabel('Accuracy (%)')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, 
             bar.get_height() + 0.3, 
             f'{acc:.2f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
# Cell 10 - Confusion Matrix

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_preds = [("Naive Bayes", nb_pred), 
               ("Logistic Regression", lr_pred), 
               ("Linear SVM", svm_pred)]

for ax, (name, pred) in zip(axes, model_preds):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Neg', 'Neu', 'Pos'],
                yticklabels=['Neg', 'Neu', 'Pos'])
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# Cell 11 - Predict Custom Sentiment

def predict_sentiment(text):
    cleaned = preprocess_text(text)
    vectorized = tfidf.transform([cleaned])
    
    label_reverse = {2: '😊 Positive', 1: '😐 Neutral', 0: '😡 Negative'}
    
    nb_result  = label_reverse[nb_model.predict(vectorized)[0]]
    lr_result  = label_reverse[lr_model.predict(vectorized)[0]]
    svm_result = label_reverse[svm_model.predict(vectorized)[0]]
    
    print(f"\n📝 Text: '{text}'")
    print(f"   Naive Bayes         : {nb_result}")
    print(f"   Logistic Regression : {lr_result}")
    print(f"   Linear SVM          : {svm_result}")

# Test examples
predict_sentiment("I absolutely love this airline! Best flight ever!")
predict_sentiment("The flight was delayed and customer service was terrible.")
predict_sentiment("Flight was okay, nothing special about it.")
predict_sentiment("Amazing experience, will definitely fly again!")
predict_sentiment("Worst experience of my life, never again!")